In [3]:
state = {
    "messages": [{'turno': 1, 'role': 'user', 'text': 'Hola, me llamo Matías.'}, {'turno': 1, 'role': 'model', 'text': '¡Hola, Matías! Es un gusto saludarte. ¿En qué puedo ayudarte el día de hoy?'}, {'turno': 2, 'role': 'user', 'text': 'Explica en 20 palabras quién fué el mejor físico del mundo.'}, {'turno': 2, 'role': 'model', 'text': 'Albert Einstein es considerado el mejor físico por revolucionar nuestra comprensión del espacio, el tiempo, la energía y la gravedad universal.'}, {'turno': 3, 'role': 'user', 'text': 'Dime en 10 palabras cuál es la mejor canción de los últimos 100 años.'}, {'turno': 3, 'role': 'model', 'text': '"Bohemian Rhapsody" de Queen es considerada la mejor canción histórica.'}, {'turno': 4, 'role': 'user', 'text': 'Reproduce con exactitud, sin añadir ni quitar nada, el turno 2 de nuestra conversación, indicando cual fue mi pregunta y cuál tu respuesta.'}, {'turno': 4, 'role': 'model', 'text': 'Pregunta: Explica en 20 palabras quién fué el mejor físico del mundo.\nRespuesta: Albert Einstein es considerado el mejor físico por revolucionar nuestra comprensión del espacio, el tiempo, la energía y la gravedad universal.'}, {'turno': 5, 'role': 'user', 'text': 'Dime cómo me llamo.'}, {'turno': 5, 'role': 'model', 'text': 'No tengo acceso a tu nombre, ya que no lo has mencionado en nuestra conversación hasta ahora.'}, {'turno': 6, 'role': 'user', 'text': 'Reproduce con exactitud, sin añadir ni quitar nada, el turno 10 de nuestra conversación, indicando cual fue mi pregunta y cuál tu respuesta.'}, {'turno': 6, 'role': 'model', 'text': 'No existe un turno 10 en nuestra conversación, ya que hasta el momento solo hemos llegado hasta el turno 5.'}]
}

In [4]:
PLANTILLA_RESUMEN = """
Eres un asistente que resume conversaciones mantenidas con un LLM dedicado a dar asistencia en el onboarding de una compañía. Vas a recibir un texto que incluye todo el historial de la conversación, indicando el número de turno de la misma, el rol de quién escribe cada mensaje (user o model) y el texto del mensaje.


Tarea: resume el texto en {max_puntos} puntos clave.
Reglas:
- No inventes ningún dato.
- Conserva nombres, temas de sobre los que se ha preguntado y preferencias del usuario.
- Devuelve solo la lista en texto plano.

El resumen debe ser óptimo para dar contexto al asistente de onboarding y que recuerde apropiadamente la conversación mantenida hasta ahora.

--- HISTORIAL DE LA CONVERSACIÓN A RESUMIR ---:
{historial}
"""

In [16]:
def historial_como_texto(state: dict) -> str:
    lines = []
    for msg in state.get("messages", []):
        lines.append(f"Turno {msg.get('turno')} - {msg.get('role', 'user')}: {msg.get('text', '')}")
    return "\n".join(lines)

hist_como_texto = historial_como_texto(state)

In [17]:
print(hist_como_texto)

Turno 1 - user: Hola, me llamo Matías.
Turno 1 - model: ¡Hola, Matías! Es un gusto saludarte. ¿En qué puedo ayudarte el día de hoy?
Turno 2 - user: Explica en 20 palabras quién fué el mejor físico del mundo.
Turno 2 - model: Albert Einstein es considerado el mejor físico por revolucionar nuestra comprensión del espacio, el tiempo, la energía y la gravedad universal.
Turno 3 - user: Dime en 10 palabras cuál es la mejor canción de los últimos 100 años.
Turno 3 - model: "Bohemian Rhapsody" de Queen es considerada la mejor canción histórica.
Turno 4 - user: Reproduce con exactitud, sin añadir ni quitar nada, el turno 2 de nuestra conversación, indicando cual fue mi pregunta y cuál tu respuesta.
Turno 4 - model: Pregunta: Explica en 20 palabras quién fué el mejor físico del mundo.
Respuesta: Albert Einstein es considerado el mejor físico por revolucionar nuestra comprensión del espacio, el tiempo, la energía y la gravedad universal.
Turno 5 - user: Dime cómo me llamo.
Turno 5 - model: No te

In [5]:
# Importar API key mediante input oculto
import os
import getpass
if not os.getenv("GEMINI_API_KEY"):
    # Pide la key en input oculto y la guarda solo en esta sesión
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Pega aquí tu GEMINI_API_KEY (input oculto): ")

print("GEMINI_API_KEY configurada:", "sí" if os.getenv("GEMINI_API_KEY") else "no")

GEMINI_API_KEY configurada: sí


In [6]:
from google import genai
from google.genai import types

# Instanciar el cliente y elegir el modelo
cliente = genai.Client()
MODEL = 'gemini-3.1-flash-lite'

In [7]:
def llamar_gemini(prompt: str) -> str:
    # Respuesta
    response = cliente.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return (response.text or "").strip()

In [ ]:
print(llamar_gemini(PLANTILLA_RESUMEN.format(
            max_puntos=5,
            historial=state
        )))

1. El usuario se identificó al inicio de la conversación como Matías.
2. El usuario ha realizado consultas sobre cultura general, específicamente sobre el mejor físico de la historia (Albert Einstein) y la mejor canción de los últimos 100 años (Bohemian Rhapsody).
3. El usuario ha mostrado interés en verificar la precisión del historial, solicitando la reproducción exacta de turnos anteriores.
4. Existe una inconsistencia en la memoria del asistente, ya que no logró recordar el nombre del usuario (Matías) proporcionado en el primer turno.
5. El usuario ha intentado acceder a información inexistente solicitando el turno 10 de la conversación, lo cual fue correctamente aclarado por el modelo.


In [18]:
print(llamar_gemini(PLANTILLA_RESUMEN.format(
            max_puntos=5,
            historial=hist_como_texto
        )))

1. El usuario se identificó al inicio de la conversación como Matías.
2. El asistente no logró retener el nombre del usuario tras ser consultado en el turno 5.
3. El usuario realizó consultas de cultura general sobre el mejor físico (Albert Einstein) y la mejor canción (Bohemian Rhapsody).
4. El usuario ha solicitado en dos ocasiones la reproducción exacta de turnos anteriores de la conversación.
5. El asistente ha demostrado capacidad para identificar correctamente los límites actuales del historial de la conversación.


In [19]:
def ultimos_n_mensajes (state: dict, n: int) -> list[dict]:
    all_messages = state.get("messages", [])
    return all_messages[-n:] if n > 0 else []

print(ultimos_n_mensajes(state, 4))

[{'turno': 5, 'role': 'user', 'text': 'Dime cómo me llamo.'}, {'turno': 5, 'role': 'model', 'text': 'No tengo acceso a tu nombre, ya que no lo has mencionado en nuestra conversación hasta ahora.'}, {'turno': 6, 'role': 'user', 'text': 'Reproduce con exactitud, sin añadir ni quitar nada, el turno 10 de nuestra conversación, indicando cual fue mi pregunta y cuál tu respuesta.'}, {'turno': 6, 'role': 'model', 'text': 'No existe un turno 10 en nuestra conversación, ya que hasta el momento solo hemos llegado hasta el turno 5.'}]


In [21]:
def ultimos_n_mensajes (state: dict, n: int) -> list[dict]:
    lines = []
    all_messages = state.get("messages", [])
    for message in all_messages[-n*2:]:
        lines.append(f"Turno {message.get('turno')} - {message.get('role', 'user')}: {message.get('text', '')}")
    return "\n".join(lines)

print(ultimos_n_mensajes(state, 4))

Turno 3 - user: Dime en 10 palabras cuál es la mejor canción de los últimos 100 años.
Turno 3 - model: "Bohemian Rhapsody" de Queen es considerada la mejor canción histórica.
Turno 4 - user: Reproduce con exactitud, sin añadir ni quitar nada, el turno 2 de nuestra conversación, indicando cual fue mi pregunta y cuál tu respuesta.
Turno 4 - model: Pregunta: Explica en 20 palabras quién fué el mejor físico del mundo.
Respuesta: Albert Einstein es considerado el mejor físico por revolucionar nuestra comprensión del espacio, el tiempo, la energía y la gravedad universal.
Turno 5 - user: Dime cómo me llamo.
Turno 5 - model: No tengo acceso a tu nombre, ya que no lo has mencionado en nuestra conversación hasta ahora.
Turno 6 - user: Reproduce con exactitud, sin añadir ni quitar nada, el turno 10 de nuestra conversación, indicando cual fue mi pregunta y cuál tu respuesta.
Turno 6 - model: No existe un turno 10 en nuestra conversación, ya que hasta el momento solo hemos llegado hasta el turno 5